# Simple Laplace Problem: Discretization with linear finite elements

We solve the Laplace problem 

$$
\begin{align*}
-\Delta u = f \text{ in }\Omega,\qquad u=0 \text{ on }\partial\Omega
\end{align*}
$$

by using linear Lagrange finite elements.

In [ ]:
from ngsolve import *
from netgen.occ import unit_square
from ngsolve.webgui import Draw

mesh = Mesh(unit_square.GenerateMesh(maxh=0.1))
Draw(mesh);

We assume a simple exact solution $u$ and compute the corresponding right-hand side $f$.

In [ ]:
# Exact solution and corresponding source term
u_ex = CF(x*(1-x)*y*(1-y))
f = -(u_ex.Diff(x).Diff(x) + u_ex.Diff(y).Diff(y))

Draw(u_ex, mesh);

The weak discrete problem is: Find $u_h\in V_h^1$ such that for all $v\in V_h^1$

$$
\begin{align*}
a(u_h,v_h) = \ell(v_h), \qquad \text{with }\qquad a(u_h,v_h) = \int_{\Omega}\nabla u_h\cdot\nabla v_h\,dx,\quad \ell(v_h) = \int_{\Omega}f\, v_h\,dx
\end{align*}
$$
and where $V_h^1$ is the space of element-wise linear polynomials that are globally continuous and zero at the boundary. $V_h^1$ is spanned by $\{\varphi_i\}$ where $\varphi_i$ are the barycentric coordinates (so-called hat functions).

With $\underline{A}_{ij}=a(\varphi_i,\varphi_j)$ and $\underline{b}_i = \ell(\varphi_i)$ we obtain the linear system

$$
\begin{align*}
\underline{A}\underline{u}=\underline{b},
\end{align*}
$$

where $\underline{u}\in\mathbb{R}^N$ is the coefficient vector with respect to the basis $\{\varphi_i\}$ for the discrete solution $u_h=\sum_{i=1}^N\underline{u}_i\varphi_i$.

In [ ]:
# Linear Lagrange finite elements with zero Dirichlet boundary conditions
Vh = H1(mesh, order=1, dirichlet=".*")

# Trial and test functions
u_h, v_h = Vh.TnT()

# Bilinear and linear forms
a = BilinearForm(Vh)
a += grad(u_h) * grad(v_h) * dx

l = LinearForm(Vh)
l += f * v_h * dx

# Assemble and solve the system
a.Assemble()
l.Assemble()
gf_u_h = GridFunction(Vh)

# Solve the linear system
gf_u_h.vec.data = a.mat.Inverse(Vh.FreeDofs(), inverse="sparsecholesky") * l.vec

Draw(gf_u_h, mesh);

print(f"L2 error = {sqrt(Integrate((u_ex - gf_u_h)**2, mesh))}")
print(f"H1 error = {sqrt(Integrate((CF((u_ex.Diff(x), u_ex.Diff(y))) - grad(gf_u_h))**2, mesh))}")

In [ ]:
gf = GridFunction(Vh)
gf.vec[:] = 0
gf.vec[Vh.ndof-1] = 1
Draw(gf, mesh, deformation=True);

gf.vec[:] = 0
gf.vec[0] = 1
Draw(gf, mesh, deformation=True);